In [2]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install fsspec

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from huggingface_hub import login

# Thay cái token của bạn vào đây
login(token="")

In [8]:
import pandas as pd

df = pd.read_json("hf://datasets/anti-ai/ViNLI-Zalo-supervised/law_vi.jsonl.gz", lines=True)

MemoryError: 

In [9]:
df

NameError: name 'df' is not defined

In [10]:
import os

embed_dir = "../../data/embed"
os.makedirs(embed_dir, exist_ok=True)

print(f"Thư mục {embed_dir} đã được tạo thành công!")
print(f"Dataset có {len(df)} dòng dữ liệu")

Thư mục ../../data/embed đã được tạo thành công!


NameError: name 'df' is not defined

In [5]:
# Lưu dataset dưới nhiều định dạng khác nhau trong thư mục data/embed

# 1. Lưu dưới dạng JSON Lines (giữ nguyên định dạng gốc)
jsonl_path = os.path.join(embed_dir, "law_vi.jsonl")
df.to_json(jsonl_path, orient='records', lines=True, force_ascii=False)
print(f"✓ Đã lưu dataset dưới dạng JSONL: {jsonl_path}")


print(f"\n📊 Tổng kết:")
print(f"- Số lượng bản ghi: {len(df):,}")
print(f"- Số cột: {len(df.columns)}")
print(f"- Cột có sẵn: {list(df.columns)}")
print(f"- Thư mục lưu trữ: {os.path.abspath(embed_dir)}")

✓ Đã lưu dataset dưới dạng JSONL: ../../data/embed\law_vi.jsonl

📊 Tổng kết:
- Số lượng bản ghi: 32,980
- Số cột: 3
- Cột có sẵn: ['query', 'positive', 'hard_neg']
- Thư mục lưu trữ: e:\MachineLearning\Legal\data\embed


In [6]:
# Kiểm tra nội dung của dataset để đảm bảo dữ liệu đã được tải chính xác
print("🔍 Xem trước dữ liệu:")
print(f"Shape: {df.shape}")
print("\nCác cột:")
for col in df.columns:
    print(f"- {col}: {df[col].dtype}")

print("\n📝 Mẫu dữ liệu đầu tiên:")
print("="*50)
for col in df.columns:
    if len(df) > 0:
        sample_value = str(df[col].iloc[0])
        if len(sample_value) > 100:
            sample_value = sample_value[:100] + "..."
        print(f"{col}: {sample_value}")
print("="*50)

🔍 Xem trước dữ liệu:
Shape: (32980, 3)

Các cột:
- query: object
- positive: object
- hard_neg: object

📝 Mẫu dữ liệu đầu tiên:
query: Tổ sát_hạch cấp giấy_phép lái tàu_hỏa có bao_nhiêu thành_viên ?
positive: Điều 30 . Tổ sát_hạch 1 . Tổ sát_hạch do Cục_trưởng Cục Đường_sắt Việt_Nam thành_lập , chịu sự chỉ_đ...
hard_neg: Điều 10 . Xác_định nội_dung đăng_ký lại khai_sinh 1 . Trường_hợp người yêu_cầu đăng_ký lại khai_sinh...


In [12]:
import json
import gzip
import requests
from pathlib import Path

# 1. Cấu hình (Thay URL đúng của file triplet nếu cần)
url = "https://huggingface.co/datasets/anti-ai/ViNLI-Zalo-supervised/resolve/main/law_vi.jsonl.gz"
out_file = Path("train_rag.jsonl")

def normalize_text(s):
    if not s: return ""
    return " ".join(str(s).replace("_", " ").split())

print("Đang nạp dữ liệu trực tiếp từ URL (Streaming)...")

# 2. Mở luồng tải (Stream) - KHÔNG nạp cả file vào RAM
with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with gzip.GzipFile(fileobj=r.raw) as gz:
        with out_file.open("w", encoding="utf-8") as fout:
            for i, line in enumerate(gz):
                try:
                    row = json.loads(line)
                    
                    # Tùy biến key theo file triplet của bạn (query/positive hoặc question/context)
                    question = normalize_text(row.get("query") or row.get("question") or "Nội dung pháp luật")
                    context = normalize_text(row.get("positive") or row.get("context") or "")
                    hard_neg = normalize_text(row.get("hard_neg", ""))

                    if not context: continue

                    out = {
                        "question": question,
                        "context": context,
                        "source": "law_vi_triplet",
                        "hard_neg": hard_neg
                    }
                    fout.write(json.dumps(out, ensure_ascii=False) + "\n")
                    
                    if i % 10000 == 0:
                        print(f"-> Đã xong {i} dòng...")
                        
                except Exception as e:
                    continue

print(f"\n[XONG] Đã lưu tại: {out_file}")

Đang nạp dữ liệu trực tiếp từ URL (Streaming)...
-> Đã xong 0 dòng...
-> Đã xong 10000 dòng...
-> Đã xong 20000 dòng...
-> Đã xong 30000 dòng...

[XONG] Đã lưu tại: train_rag.jsonl
